# MIMIC-CLIF Early PT Consults Comparative Analysis

- Uses MIMIC-CLIF dataset for plans of future roll out to CLIF however, it collects missing data from MIMIC directly.
- This is mostly collecting the data and aggregating it in preparation for data analysis which happens separately.

In [1]:
import pandas as pd
from datetime import datetime
from datetime import timedelta
import numpy as np
import pthelperfunctions as helper
import os

#Load block data CLIF-Eligibility-for-mobilization output
block_df = helper.load_data('mob','cohort_all_ids_w_outcome',folder='intermediate')
block_df['encounter_block'] = block_df['encounter_block'].astype(str)

#Add the output with the yellow time to eligibility without business hours.
yellow_df = helper.load_data('mob','competing_risk_yellow_final_all_hours',folder='intermediate')
yellow_df['encounter_block'] = yellow_df['encounter_block'].astype(str)
yellow_df = yellow_df[['encounter_block','time_eligibility']].rename(columns={'time_eligibility':'yellow_time_eligibility'})
block_df = pd.merge(
    block_df,
    yellow_df,
    on='encounter_block',
    how='left'
)

####TESTING ONLY####################################
#block_df = block_df.head(100) #smallercohort for development
#print("WARNING: Smaller cohort size for testing.")

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")
print(f"Unique Admission IDs: {block_df['hospitalization_id'].nunique()}")

📢 ClifOrchestrator initialized
Block Length: 32147
Unique Encounter Block: 32117
Unique Admission IDs: 32127


## Merging strategy

All of the CLIF tables are associated with a hospitalization_id without any patient_id data. We have these two data frames below.


- block_df (From the CLIF Eligibility for Mobilization Output)
    - encounter_block, hospitalization_id, patient_id

- hosp_df (CLIF)
    - hospitalization_id, patient_id

MERGE hosp_df and block_df on=patient how=left.

NEW hosp_df
- hospitalization_id, patient_id, encounter_block
Note this would create duplicates, which is on purpose.

Now we can look at any CLIF table.
For a given encounter_block, it will pull data from any hospitalization_id associated with the same patient in the encounter_block.
Then we can filter based on date_time only wihtout any sort of filter based on hospitalization_id.

## Hospitalization

In [2]:
#Load and filter Hospitalizations data from CLIF
hosp_df = helper.load_clif_table('hospitalization')
hosp_df = hosp_df[hosp_df['patient_id'].isin(block_df['patient_id'])].reset_index()
print(f"Hosp DF size: {len(hosp_df)}")

#Add encounter_block to the hospitalization data frame.
hosp_df = hosp_df.sort_values('admission_dttm')
hosp_df = hosp_df.merge(
    block_df[['patient_id','encounter_block','block_vent_start_dttm']].drop_duplicates(),
    on='patient_id',
    how='left').reset_index()

hosp_list = hosp_df['hospitalization_id'].drop_duplicates().tolist()

print(f"Hosp DF size: {len(hosp_df)}")

📢 Initialized hospitalization table
📢 Data directory: /nfs/roberts/project/pi_sj692/shared/mimicclif
📢 File type: parquet
📢 Timezone: America/New_York
📢 Output directory: /nfs/roberts/project/pi_sj692/shared/PT_consults/output
📢 Loaded schema from /nfs/roberts/project/pi_sj692/gcr22/clifenv/lib/python3.12/site-packages/clifpy/schemas/hospitalization_schema.yaml
📢 Loaded outlier configuration
📢 Initialized hospitalization table
📢 Data directory: /nfs/roberts/project/pi_sj692/shared/mimicclif
📢 File type: parquet
📢 Timezone: America/New_York
📢 Output directory: /nfs/roberts/project/pi_sj692/shared/PT_consults/output
📢 Loaded schema from /nfs/roberts/project/pi_sj692/gcr22/clifenv/lib/python3.12/site-packages/clifpy/schemas/hospitalization_schema.yaml
📢 Loaded outlier configuration
Using CLIF standard outlier ranges

Building outlier expressions...


Building expressions: 100%|██████████| 1/1 [00:00<00:00, 3125.41column/s]


Applying outlier filtering...


Processing: 100%|██████████| 1/1 [00:00<00:00, 468.95operation/s]


age_at_admission              : 546028 values →      0 nullified (  0.0%)
Hosp DF size: 104664
Hosp DF size: 137376


In [3]:
# Add admit_dttm and admit_type_name from hospitalization table

#Merge back to block_df
block_df = pd.merge(
    block_df,
    hosp_df[["patient_id", "hospitalization_id", "admission_dttm","admission_type_category"]].drop_duplicates(),
    on=["patient_id", "hospitalization_id"],
    how='left'
)

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}") 

Block Length: 32147
Unique Encounter Block: 32117


Unfortunately the CLIF admission_type_category mapping from MIMIC-CLIF does not include whether or not the patient came as an OSH transfer. So we will need to get data direct from MIMIC to determine OSH transfers.

In [4]:
if helper.use_mimic:
    mimic_hosp_df = helper.load_data('mimic', 'admissions',folder='hosp',type='csv.gz')
    mimic_hosp_df.rename(columns={'hadm_id':'hospitalization_id','subject_id':'patient_id'}, inplace=True)
    mimic_hosp_df['hospitalization_id'] = mimic_hosp_df['hospitalization_id'].astype(str)
    block_df = block_df.merge(
        mimic_hosp_df[['hospitalization_id','admission_location']],
        on='hospitalization_id',
        how='left')
    block_df['admission_osh'] = (block_df['admission_location'] == 'TRANSFER FROM HOSPITAL').astype(bool)

## ICU ADT

In [5]:
#ICU ADT data
#Load the CLIF tables
adt_df = helper.load_clif_table('adt')

#Merge with encounter block
merged_adt_df = pd.merge (
    block_df[['encounter_block','hospitalization_id','block_vent_start_dttm']],
    adt_df,
    on='hospitalization_id',
    how = 'inner'
)
#keep ICUs only
merged_adt_df = merged_adt_df[merged_adt_df['location_category']== 'icu']

#Sort by in_dttm
merged_adt_df= merged_adt_df.sort_values(['encounter_block','in_dttm'], ascending=True)

#ICU Type and In Time
#remove anywhere the patient left before start of IMV
ICU_df = merged_adt_df[merged_adt_df['out_dttm'] > merged_adt_df['block_vent_start_dttm']].copy()
ICU_df = ICU_df.drop_duplicates(subset=['encounter_block'], keep='first')
ICU_df.rename(columns={'location_name':'ICU_type','in_dttm':'icu_in_dttm','out_dttm':'icu_out_dttm'}, inplace=True)
ICU_df = ICU_df[['encounter_block','ICU_type','icu_in_dttm','icu_out_dttm']]
#Merge back to block
block_df = block_df.merge(
    ICU_df,
    on='encounter_block',
    how='left'
)
block_df['ICU_type'] = block_df['ICU_type'].astype(str)


#ICU Re-admission, post IMV
ICU_df = merged_adt_df[merged_adt_df['out_dttm'] > merged_adt_df['block_vent_start_dttm']]
ICU_df['icu_readmission'] = ICU_df['encounter_block'].duplicated(keep='first') #If an encounter block is duplicated after removing pre-IMV ICU stays, this implies re-admission.
icu_readmit_df = (
   ICU_df.groupby(["encounter_block"], as_index=False)
    .agg(icu_readmission=("icu_readmission", "max"))
)
#Merge back to blocks
block_df = block_df.merge(
    icu_readmit_df[["encounter_block", "icu_readmission"]],
    on=["encounter_block"],
    how="left"
)
block_df["icu_readmission"] = block_df["icu_readmission"].astype(bool)

#ICU stays in total
#Whole encounter block, not just post-IMV
icu_N_df = (
    merged_adt_df.groupby(["encounter_block"], as_index=False)
    .agg(icu_N=("in_dttm", "count"))
)
#Marge back to blocks
block_df = block_df.merge(
    icu_N_df[["encounter_block", "icu_N"]],
    on=["encounter_block"],
    how="left"
)

#ICU LOS, including whole encounter block, not just post IMV
merged_adt_df["icu_los_days"] = (merged_adt_df["out_dttm"] - merged_adt_df["in_dttm"]).dt.total_seconds() / (24*3600)
icu_los_df = (
    merged_adt_df.groupby(["encounter_block"], as_index=False)
    .agg(icu_los_days=("icu_los_days", "sum"))
)
#Merge back to blocks
block_df = block_df.merge(
    icu_los_df[["encounter_block", "icu_los_days"]],
    on=["encounter_block"],
    how="left"
)

del adt_df, merged_adt_df, ICU_df
print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")


📢 Initialized adt table
📢 Data directory: /nfs/roberts/project/pi_sj692/shared/mimicclif
📢 File type: parquet
📢 Timezone: America/New_York
📢 Output directory: /nfs/roberts/project/pi_sj692/shared/PT_consults/output
📢 Loaded schema from /nfs/roberts/project/pi_sj692/gcr22/clifenv/lib/python3.12/site-packages/clifpy/schemas/adt_schema.yaml
📢 Loaded outlier configuration
📢 Initialized adt table
📢 Data directory: /nfs/roberts/project/pi_sj692/shared/mimicclif
📢 File type: parquet
📢 Timezone: America/New_York
📢 Output directory: /nfs/roberts/project/pi_sj692/shared/PT_consults/output
📢 Loaded schema from /nfs/roberts/project/pi_sj692/gcr22/clifenv/lib/python3.12/site-packages/clifpy/schemas/adt_schema.yaml
📢 Loaded outlier configuration
Using CLIF standard outlier ranges

No outlier configuration found for table: adt
Block Length: 32147
Unique Encounter Block: 32117


/tmp/ipykernel_2287730/1392376350.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ICU_df['icu_readmission'] = ICU_df['encounter_block'].duplicated(keep='first') #If an encounter block is duplicated after removing pre-IMV ICU stays, this implies re-admission.


## Demographics

In [6]:
## FOR Sex, Race and Ethnicity, and Language

#Load and Filter Patient DF from CLIF
patient_df = helper.load_clif_table('patient')
patient_df = patient_df[patient_df['patient_id'].isin ( block_df['patient_id'] )].reset_index()
patient_df = patient_df[['patient_id','sex_category','race_category','ethnicity_category','birth_date','language_category']]

#merge
block_df = pd.merge(
    block_df,
    patient_df,
    on='patient_id',
    how='left'
)

#Convert birth date to date object
if not helper.use_mimic:
    block_df['birth_date'] = pd.to_date(block_df['birth_date'], errors='coerce')
    block_df['age'] = block_df['block_vent_start_dttm'].dt.year - block_df['birth_date'].dt.year

#remove useless column
block_df = block_df.drop(columns=['birth_date'])

del patient_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

📢 Initialized patient table
📢 Data directory: /nfs/roberts/project/pi_sj692/shared/mimicclif
📢 File type: parquet
📢 Timezone: America/New_York
📢 Output directory: /nfs/roberts/project/pi_sj692/shared/PT_consults/output
📢 Loaded schema from /nfs/roberts/project/pi_sj692/gcr22/clifenv/lib/python3.12/site-packages/clifpy/schemas/patient_schema.yaml
📢 Loaded outlier configuration
📢 Initialized patient table
📢 Data directory: /nfs/roberts/project/pi_sj692/shared/mimicclif
📢 File type: parquet
📢 Timezone: America/New_York
📢 Output directory: /nfs/roberts/project/pi_sj692/shared/PT_consults/output
📢 Loaded schema from /nfs/roberts/project/pi_sj692/gcr22/clifenv/lib/python3.12/site-packages/clifpy/schemas/patient_schema.yaml
📢 Loaded outlier configuration
Using CLIF standard outlier ranges

No outlier configuration found for table: patient
Block Length: 32147
Unique Encounter Block: 32117


##AGE, LANGUAGE, Payor from MIMIC data given missing on CLIF'd tables.

Language and Payor for MIMIC is extracted from the admissions table.
Our CLIF'd tables are missing birth date since this is not in MIMIC but rather uses anchor_year and anchor_age.
We are calculating age at the time of intubation.
Also, note that MIMIC age stops at 89, all patients older than 89 are assiged anchor_age 91.
Language for MIMIC is extracted from the admissions table.


In [7]:
##AGE, LANGUAGE, Payor from MIMIC data given missing on CLIF'd tables.

if helper.use_mimic:
    #AGE
    
    #load
    patient_mimic_df = helper.load_data("mimic","patients", folder='hosp', type='csv.gz')
    patient_mimic_df.rename(columns={'subject_id': 'patient_id'}, inplace=True)
    patient_mimic_df['patient_id'] = patient_mimic_df['patient_id'].astype(str)
    
    #filter
    patient_mimic_df = patient_mimic_df[patient_mimic_df['patient_id'].isin( block_df['patient_id'] )]
    print(f"Unique MIMIC patient id: {patient_mimic_df['patient_id'].nunique()}")
    print(f"Unique Block patient id: {block_df['patient_id'].nunique()}")
    
    #merge
    merged_patient_df = pd.merge(
        block_df,
        patient_mimic_df,
        on='patient_id',
        how='left'
    )
    #convert admission time to year and calculate year - anchor_year + anchor_age
    merged_patient_df['age'] = merged_patient_df['block_vent_start_dttm'].dt.year - merged_patient_df['anchor_year'] + merged_patient_df['anchor_age']

    #Date of death
    merged_patient_df['date_of_death'] = pd.to_datetime(merged_patient_df['dod'])
    merged_patient_df['date_of_death'] = helper.ensure_datetime_est(merged_patient_df['date_of_death'])

    #Admission Year (based on estimate from anchor year
    merged_patient_df["anchor_year_group"] = merged_patient_df["anchor_year_group"].str.slice(0, 4).astype(int) + 1 #Convert achor year group to an integer
    merged_patient_df["admission_year"] = block_df['block_first_vital_dttm'].dt.year.astype(int) - merged_patient_df["anchor_year"] + merged_patient_df["anchor_year_group"]
    
    #drop unnecessary columns
    block_df = merged_patient_df.drop(columns=['gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod'])
    
    ###LANGUAGE & PAYOR
    
    #For Language, we need to get this from a different table.
    block_df = block_df.drop(columns=['language_category'])
    
    #load
    adm_mimic_df = helper.load_data("mimic","admissions", folder='hosp', type='csv.gz')
    adm_mimic_df.rename(columns={'hadm_id': 'hospitalization_id'}, inplace=True)
    adm_mimic_df['hospitalization_id'] = adm_mimic_df['hospitalization_id'].astype(str)
    adm_mimic_df['mimic_race'] = adm_mimic_df['race'].astype(str)
    adm_mimic_df['insurance'] = adm_mimic_df['insurance'].astype(str)
    adm_mimic_df['language'] = adm_mimic_df['language'].astype(str)
    adm_mimic_df = adm_mimic_df[['hospitalization_id','language','insurance','mimic_race']]
    
    #filter
    adm_mimic_df = adm_mimic_df[adm_mimic_df['hospitalization_id'].isin( block_df['hospitalization_id'] )]
    print(f"Unique MIMIC hosp id: {adm_mimic_df['hospitalization_id'].nunique()}")
    print(f"Unique Block patient id: {block_df['hospitalization_id'].nunique()}")
    
    #merge
    block_df = pd.merge(
        block_df,
        adm_mimic_df,
        on='hospitalization_id',
        how='left'
    )

    #print("MIMIC Race:")
    #print(block_df['mimic_race'].value_counts())

    del patient_mimic_df, merged_patient_df, adm_mimic_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Unique MIMIC patient id: 29390
Unique Block patient id: 29390
Unique MIMIC hosp id: 32127
Unique Block patient id: 32127
Block Length: 32147
Unique Encounter Block: 32117


## PT Order
From MIMIC chart events

In [8]:
#load (loading from output since key_icu_orders is not a defined table in CLIFpy and we just created it in the prior script
pt_df = helper.load_data("output","clif_key_icu_orders")

#Filter for PT orders only
chart_mask = pt_df["order_category"].isin(['pt_evaluation','pt_treat'])
pt_df = pt_df[chart_mask]

#Merge with hosp_df
merged_pt_df = pt_df.merge(hosp_df[['hospitalization_id','encounter_block','block_vent_start_dttm']],on='hospitalization_id', how='right').reset_index()
merged_pt_df['time_diff'] = merged_pt_df['order_dttm'] - merged_pt_df['block_vent_start_dttm']

#PT pre IMV
pt_pre_imv = merged_pt_df[merged_pt_df['time_diff'].dt.total_seconds() < 0]
pt_pre_imv = pt_pre_imv.groupby('encounter_block')['order_dttm'].agg('max').reset_index()
pt_pre_imv.rename(columns={'order_dttm':'pt_pre_imv_dttm'}, inplace=True)
block_df = pd.merge(
    block_df,
    pt_pre_imv,
    on='encounter_block',
    how='left'
)

#PT post IMV
pt_post_imv = merged_pt_df[merged_pt_df['time_diff'].dt.total_seconds() >= 0]
pt_post_imv = pt_post_imv.groupby('encounter_block')['order_dttm'].agg('min').reset_index()
pt_post_imv.rename(columns={'order_dttm':'pt_post_imv_dttm'}, inplace=True)
block_df = pd.merge(
    block_df,
    pt_post_imv,
    on='encounter_block',
    how='left'
)

del chart_mask, pt_df, merged_pt_df, pt_pre_imv, pt_post_imv

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")


Block Length: 32147
Unique Encounter Block: 32117


## Vitals

We have looked into pulling vitals data from the MIMIC ED database to see if it improves missingness of pre-intubations vitals.
It added pre-intubation vital signs for about 10% of encounters but we still had 40% missingness so this portion of the code was removed.
It can still be found in old versions of the code as needed.

In [9]:
#Load the Vitals CLIF tables
vitals_df = helper.load_clif_table('vitals', hosp_list=hosp_list)

#Remove the ones not in cohort, add encounter_block and vent start time as reference
merged_vitals_df = vitals_df.merge(hosp_df[['hospitalization_id','encounter_block','block_vent_start_dttm']],on='hospitalization_id', how='right').reset_index()

#Calculate time windows from block_vent_start_dttm
merged_vitals_df['time_diff'] = merged_vitals_df['recorded_dttm'] - merged_vitals_df['block_vent_start_dttm']

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

📢 Initialized vitals table
📢 Data directory: /nfs/roberts/project/pi_sj692/shared/mimicclif
📢 File type: parquet
📢 Timezone: America/New_York
📢 Output directory: /nfs/roberts/project/pi_sj692/shared/PT_consults/output
📢 Loaded schema from /nfs/roberts/project/pi_sj692/gcr22/clifenv/lib/python3.12/site-packages/clifpy/schemas/vitals_schema.yaml
📢 Loaded outlier configuration


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

📢 Initialized vitals table
📢 Data directory: /nfs/roberts/project/pi_sj692/shared/mimicclif
📢 File type: parquet
📢 Timezone: America/New_York
📢 Output directory: /nfs/roberts/project/pi_sj692/shared/PT_consults/output
📢 Loaded schema from /nfs/roberts/project/pi_sj692/gcr22/clifenv/lib/python3.12/site-packages/clifpy/schemas/vitals_schema.yaml
📢 Loaded outlier configuration
Using CLIF standard outlier ranges

Building outlier expressions...


Building expressions: 100%|██████████| 1/1 [00:00<00:00, 2861.05column/s]


Applying outlier filtering...


Processing: 100%|██████████| 1/1 [00:04<00:00,  4.62s/operation]



Vitals Table - Category Statistics:
  dbp                 : 8840205 values →   1122 nullified (  0.0%)
  heart_rate          : 8752069 values →     37 nullified (  0.0%)
  height_cm           :  43616 values →    248 nullified (  0.6%)
  map                 : 8846843 values →   8831 nullified (  0.1%)
  respiratory_rate    : 8636655 values →   1815 nullified (  0.0%)
  sbp                 : 8850088 values →    223 nullified (  0.0%)
  spo2                : 8567015 values →   3251 nullified (  0.0%)
  temp_c              : 2446467 values →   2496 nullified (  0.1%)
  weight_kg           : 542622 values →   1485 nullified (  0.3%)


In [10]:
##HEART RATE

#Filter vitals_df for heart_rate only and merge
hr_df = merged_vitals_df[merged_vitals_df['vital_category'] == 'heart_rate'].copy()
hr_df.rename(columns={'vital_value':'hr'}, inplace=True)

#0 to 4 hour HR mean
block_df = pd.merge(
    block_df,
    helper.aggregate_by_time(hr_df,
                             'hr',
                             0,4,
                             agg_func='mean'),
    on='encounter_block',
    how='left'
)

#0 to 24 hour HR mean
block_df = pd.merge(
    block_df,
    helper.aggregate_by_time(hr_df,
                             'hr',
                             0,24,
                             agg_func='mean'),
    on='encounter_block',
    how='left'
)

del hr_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32147
Unique Encounter Block: 32117


In [11]:
print(merged_vitals_df['time_diff'].dtype)

timedelta64[ns]


In [12]:
##RESPIRATORY RATE
#Used to calculate qSOFA so will take worse (max) value.

#Filter vitals_df for respiratory_rate only and merge
rr_df = merged_vitals_df[merged_vitals_df['vital_category'] == 'respiratory_rate'].copy()
rr_df.rename(columns={'vital_value':'resp_rate'}, inplace=True)

#0 to 4 hour RR max
block_df = pd.merge(
    block_df,
    helper.aggregate_by_time(rr_df,
                             'resp_rate',
                             0,4,
                             agg_func='max'),
    on='encounter_block',
    how='left'
)

#0 to 24 hour RR max
block_df = pd.merge(
    block_df,
    helper.aggregate_by_time(rr_df,
                             'resp_rate',
                             0,24,
                             agg_func='max'),
    on='encounter_block',
    how='left'
)

del rr_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32147
Unique Encounter Block: 32117


In [13]:
##MAP

#First create MAP out of SBP and DBP when MAP is missing.

#Filter vitals_df to only relevant categories
bp_df = merged_vitals_df[
    merged_vitals_df['vital_category'].isin(['sbp', 'dbp', 'map'])
].copy()

#Pivot to get sbp, dbp, and map in columns per EB + recorded_dttm
pivoted_bp_df = bp_df.pivot_table(
    index=['encounter_block', 'time_diff'],
    columns='vital_category',
    values='vital_value',
    aggfunc='first'  # in case there are multiple values per timestamp, take the first
).reset_index()

#Identify rows where MAP is missing but SBP and DBP are present
mask_missing_map = (
    pivoted_bp_df['map'].isna() &
    pivoted_bp_df['sbp'].notna() &
    pivoted_bp_df['dbp'].notna()
)

#Compute synthetic MAP
synthetic_map_rows = pivoted_bp_df[mask_missing_map].copy()
synthetic_map_rows['vital_value'] = (
    synthetic_map_rows['sbp'] + 2 * synthetic_map_rows['dbp']
) / 3
synthetic_map_rows['vital_category'] = 'map'

#Keep only necessary columns for synthetic rows
synthetic_map_rows = synthetic_map_rows[[
    'encounter_block', 'time_diff', 'vital_category', 'vital_value'
]]

#Get existing 'map' rows (real ones)
real_map_rows = bp_df[
    bp_df['vital_category'] == 'map'
][[
    'encounter_block', 'time_diff', 'vital_category', 'vital_value'
]]

#Combine real and synthetic MAP rows
map_df = pd.concat([real_map_rows, synthetic_map_rows], ignore_index=True)
map_df.rename(columns={'vital_value':'map'}, inplace=True)

#0 to 4 hour map mean
block_df = pd.merge(
    block_df,
    helper.aggregate_by_time(map_df,
                             'map',
                             0,4,
                             agg_func='mean'),
    on='encounter_block',
    how='left'
)

#0 to 24 hour map mean
block_df = pd.merge(
    block_df,
    helper.aggregate_by_time(map_df,
                             'map',
                             0,24,
                             agg_func='mean'),
    on='encounter_block',
    how='left'
)

del bp_df, pivoted_bp_df, synthetic_map_rows, real_map_rows, map_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32147
Unique Encounter Block: 32117


In [14]:
#BMI

#Filter vitals table for height and weight only
merged_bmi_df = merged_vitals_df[
    merged_vitals_df['vital_category'].isin(['height_cm', 'weight_kg'])
].copy()


# Define whether measurement is before or after vent_start_time
merged_bmi_df['before_vent_start'] = (merged_bmi_df['time_diff'].dt.total_seconds() <= 0).astype(int)

# Calculate absolute time difference
merged_bmi_df['abs_time_diff'] = merged_bmi_df['time_diff'].abs()

# Sort data to prioritize measurements before vent start and closest in time
merged_bmi_df = merged_bmi_df.sort_values(['encounter_block', 'vital_category', 'before_vent_start', 'abs_time_diff'], 
                                    ascending=[True, True, False, True])

# Drop duplicates to keep the closest measurement for each vital_category per encounter block
merged_bmi_df = merged_bmi_df.drop_duplicates(subset=['encounter_block', 'vital_category'], keep='first')

# Pivot to get height and weight per encounter block
pivot_bmi_df = merged_bmi_df.pivot(index='encounter_block', 
                                    columns='vital_category', 
                                    values='vital_value'
                                    ).reset_index()

# Calculate BMI
pivot_bmi_df['bmi'] = pivot_bmi_df['weight_kg'] / ((pivot_bmi_df['height_cm'] / 100) ** 2)

#Add back to block
block_df = pd.merge(
    block_df,
    pivot_bmi_df,
    on='encounter_block',
    how='left'
)

del merged_bmi_df, pivot_bmi_df, vitals_df, merged_vitals_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32147
Unique Encounter Block: 32117


## Respiratory Support Table

In [15]:
#Load respiratory CLIF tables
resp_df = helper.load_clif_table('respiratory_support',hosp_list=hosp_list)

#Remove the ones not in cohort and other not useful columns, add encounter_block and vent start time as reference
merged_resp_df = resp_df.merge(hosp_df[['hospitalization_id','encounter_block','block_vent_start_dttm']],on='hospitalization_id', how='right').reset_index()

#Calculate time diff
merged_resp_df['time_diff'] = merged_resp_df['recorded_dttm'] - merged_resp_df['block_vent_start_dttm']

📢 Initialized respiratory_support table
📢 Data directory: /nfs/roberts/project/pi_sj692/shared/mimicclif
📢 File type: parquet
📢 Timezone: America/New_York
📢 Output directory: /nfs/roberts/project/pi_sj692/shared/PT_consults/output
📢 Loaded schema from /nfs/roberts/project/pi_sj692/gcr22/clifenv/lib/python3.12/site-packages/clifpy/schemas/respiratory_support_schema.yaml
📢 Loaded outlier configuration
📢 Initialized respiratory_support table
📢 Data directory: /nfs/roberts/project/pi_sj692/shared/mimicclif
📢 File type: parquet
📢 Timezone: America/New_York
📢 Output directory: /nfs/roberts/project/pi_sj692/shared/PT_consults/output
📢 Loaded schema from /nfs/roberts/project/pi_sj692/gcr22/clifenv/lib/python3.12/site-packages/clifpy/schemas/respiratory_support_schema.yaml
📢 Loaded outlier configuration
Using CLIF standard outlier ranges

Building outlier expressions...


Building expressions: 100%|██████████| 17/17 [00:00<00:00, 47220.64column/s]


Applying outlier filtering...


Processing: 100%|██████████| 1/1 [00:00<00:00, 20.63operation/s]


fio2_set                      : 1140440 values →     24 nullified (  0.0%)
lpm_set                       : 788731 values →   1685 nullified (  0.2%)
tidal_volume_set              : 429326 values →   3041 nullified (  0.7%)
resp_rate_set                 : 455420 values →    149 nullified (  0.0%)
pressure_control_set          :  12290 values →      2 nullified (  0.0%)
pressure_support_set          : 412517 values →     15 nullified (  0.0%)
flow_rate_set                 : 363217 values →   3032 nullified (  0.8%)
peak_inspiratory_pressure_set :  14446 values →      1 nullified (  0.0%)
inspiratory_time_set          : 422123 values →     37 nullified (  0.0%)
peep_set                      : 874778 values →     48 nullified (  0.0%)
tidal_volume_obs              : 893070 values →  10719 nullified (  1.2%)
resp_rate_obs                 : 786359 values →     36 nullified (  0.0%)
plateau_pressure_obs          : 292184 values →     29 nullified (  0.0%)
peak_inspiratory_pressure_obs : 78965

In [16]:
#FiO2
fio2_df = merged_resp_df.copy()

#Set all values to be from 0 to 1
def normalize_fio2(x):
    if 20 < x < 100:
        return x/100
    elif 0.2 < x < 1:
        return x
    else:
        return np.nan
fio2_df['fio2_set'] = fio2_df['fio2_set'].apply(normalize_fio2)

#Fill in missing FiO2 for Other Modes
def synthetic_fio2(given, lpm, mode):
    #Delivery mode is more important than set FiO2
    #Room air is 21%
    if mode == "Room Air":
        return 0.21
    #NC is 20% + 4%xFlow if flow is given and within range. Otherwise it is NA.
    elif mode == "Nasal Cannula":
        if 0.5 < lpm <= 6:
            return 0.2 + 0.04*lpm
        else:
            return np.nan
    #Face mask, will have to assume simple since they do not mention NRB. Set to 5% + 5% x Flow but only if flow is within reasonable range. Else return
    elif mode == "Face Mask":
        if given:
            return given
        if 5 <= lpm <= 10:
            return 0.05 + 0.05*lpm
        else:
            return np.nan
    #Trach collar will assume room air if no specific FiO2 is given.
    elif mode == "Trach Collar":
        if given:
            return given
        else:
            return 0.21
    else: #This would include IMV and NIPPV.
        return given
fio2_df['fio2_set'] = fio2_df.apply(lambda row: synthetic_fio2(row['fio2_set'],row['lpm_set'],row['device_category']), axis=1)

#0 to 4 hour FiO2 mean
block_df = pd.merge(
    block_df,
    helper.aggregate_by_time(fio2_df,
                             'fio2_set',
                             0,4,
                             agg_func='mean'),
    on='encounter_block',
    how='left'
)

#0 to 24 hour FiO2 mean
block_df = pd.merge(
    block_df,
    helper.aggregate_by_time(fio2_df,
                             'fio2_set',
                             0,24,
                             agg_func='mean'),
    on='encounter_block',
    how='left'
)

del fio2_df

In [17]:
#PEEP set

#0 to 4 hour PEEP mean
block_df = pd.merge(
    block_df,
    helper.aggregate_by_time(merged_resp_df,
                             'peep_set',
                             0,4,
                             agg_func='mean'),
    on='encounter_block',
    how='left'
)

#0 to 24 hour PEEP mean
block_df = pd.merge(
    block_df,
    helper.aggregate_by_time(merged_resp_df,
                             'peep_set',
                             0,24,
                             agg_func='mean'),
    on='encounter_block',
    how='left'
)

del merged_resp_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32147
Unique Encounter Block: 32117


## Patient Assessments

In [18]:
#Load Assessment CLIF tables
ass_df = helper.load_clif_table("patient_assessments", hosp_list=hosp_list)
ass_df = ass_df[ass_df['assessment_category'].isin(['braden_mobility','RASS','gcs_total','cam_total'])] #Make it smaller by using only the categories we want.

#Remove the ones not in cohort, add encounter_block and vent start time as reference
merged_ass_df = ass_df.merge(block_df[['hospitalization_id','encounter_block','block_vent_start_dttm','icu_out_dttm']],on='hospitalization_id', how='right').reset_index()

#Calculate time difference from block_vent_start_dttm
merged_ass_df['time_diff'] = merged_ass_df['recorded_dttm'] - merged_ass_df['block_vent_start_dttm']

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

📢 Initialized patient_assessments table
📢 Data directory: /nfs/roberts/project/pi_sj692/shared/mimicclif
📢 File type: parquet
📢 Timezone: America/New_York
📢 Output directory: /nfs/roberts/project/pi_sj692/shared/PT_consults/output
📢 Loaded schema from /nfs/roberts/project/pi_sj692/gcr22/clifenv/lib/python3.12/site-packages/clifpy/schemas/patient_assessments_schema.yaml
📢 Loaded outlier configuration


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

📢 Initialized patient_assessments table
📢 Data directory: /nfs/roberts/project/pi_sj692/shared/mimicclif
📢 File type: parquet
📢 Timezone: America/New_York
📢 Output directory: /nfs/roberts/project/pi_sj692/shared/PT_consults/output
📢 Loaded schema from /nfs/roberts/project/pi_sj692/gcr22/clifenv/lib/python3.12/site-packages/clifpy/schemas/patient_assessments_schema.yaml
📢 Loaded outlier configuration
Using CLIF standard outlier ranges

Building outlier expressions...


Building expressions: 100%|██████████| 1/1 [00:00<00:00, 855.63column/s]


Applying outlier filtering...


Processing: 100%|██████████| 1/1 [00:04<00:00,  4.94s/operation]



Patient Assessments Table - Category Statistics:
  RASS                : 1656001 values →      0 nullified (  0.0%)
  braden_activity     : 1067495 values →      0 nullified (  0.0%)
  braden_friction     : 1063850 values →      0 nullified (  0.0%)
  braden_mobility     : 1067493 values →      0 nullified (  0.0%)
  braden_moisture     : 1068127 values →      0 nullified (  0.0%)
  braden_nutrition    : 1067041 values →      0 nullified (  0.0%)
  braden_sensory      : 1068778 values →      0 nullified (  0.0%)
  braden_total        : 1061624 values →      0 nullified (  0.0%)
  cam_inattention     :      0 values →      0 nullified (  0.0%)
  cam_loc             :      0 values →      0 nullified (  0.0%)
  cam_mental          :      0 values →      0 nullified (  0.0%)
  cam_thinking        :      0 values →      0 nullified (  0.0%)
  cam_total           :      0 values →      0 nullified (  0.0%)
  gcs_eye             : 2214694 values →      0 nullified (  0.0%)
  gcs_motor      

In [19]:
#Braden Mobility Score

#Filter ass_df for braden_mob only
braden_df = merged_ass_df[merged_ass_df['assessment_category'] == 'braden_mobility'].copy()
braden_df.rename(columns={'numerical_value':'braden'}, inplace=True)

#0 to 24 hour Braden Mobility max
block_df = pd.merge(
    block_df,
    helper.aggregate_by_time(braden_df,
                             'braden',
                             0,24,
                             agg_func='max'),
    on='encounter_block',
    how='left'
)

#Redefine time-diff to look at last braden within 24 hours of ICU discharge.
#This will give the max braden within 24 hours of ICU discharge.
braden_df['time_diff'] =braden_df['recorded_dttm'] - braden_df['icu_out_dttm']
block_df = pd.merge(
    block_df,
    helper.aggregate_by_time(braden_df,
                             'braden',
                             -24,0,
                             agg_func='max'),
    on='encounter_block',
    how='left'
)
block_df.rename(columns={'braden_-24_0h_max':'braden_last'},inplace=True)

del braden_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32147
Unique Encounter Block: 32117


In [20]:
#RASS

#Filter ass_df for RASS only and merge
rass_df = merged_ass_df[merged_ass_df['assessment_category'] == 'RASS'].copy()
rass_df.rename(columns={'numerical_value':'RASS'}, inplace=True)

#0 to 24 hour RASS min
block_df = pd.merge(
    block_df,
    helper.aggregate_by_time(rass_df,
                             'RASS',
                             0,24,
                             agg_func='min'),
    on='encounter_block',
    how='left'
)

del rass_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32147
Unique Encounter Block: 32117


In [21]:
#CAM-ICU

#Filter ass_df for CAM-ICU only
cam_df = merged_ass_df[merged_ass_df['assessment_category'] == 'cam_total'].copy()
cam_df['cam_icu'] = np.where(cam_df['categorical_value'] == 'Positive',1,0)

#0 to 24 hour CAM-ICU max (of a boolean value)
block_df = pd.merge(
    block_df,
    helper.aggregate_by_time(cam_df,
                             'cam_icu',
                             0,24,
                             agg_func='max'),
    on='encounter_block',
    how='left'
)

block_df['cam_icu_0_24h_max'] = block_df['cam_icu_0_24h_max'].astype(bool)

del cam_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32147
Unique Encounter Block: 32117


In [22]:
#GCS

#Filter ass_df for GCS only and merge
gcs_df = merged_ass_df[merged_ass_df['assessment_category'] == 'gcs_total'].copy()
gcs_df.rename(columns={'numerical_value':'gcs'}, inplace=True)

#0 to 24 hour GCS min
block_df = pd.merge(
    block_df,
    helper.aggregate_by_time(gcs_df,
                             'gcs',
                             0,24,
                             agg_func='min'),
    on='encounter_block',
    how='left'
)

del gcs_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32147
Unique Encounter Block: 32117


In [23]:
#FIRST SAVING POINT
#sort of
path = os.path.join(helper.path_out, "block_compiled_1a.parquet")
block_df.to_parquet(path)
del path

## SOFA

This function uses polars dataframes instead of Pandas so we need to convert types.
The version using pandas dataframes data requires the CLIF encounter stitching to be done by clifpy and stored internally but we have our own, prepopulated encounter stitching.

In [ ]:
#SOFA scores
#Will need to use the CLIFPy SOFA Score python package.
import pandas as pd
import os
import pthelperfunctions as helper
from clifpy import compute_sofa_polars
import polars as pl
import numpy as np

block_df = helper.load_data('output','block_compiled_1a')

def my_sofa_calculator(hour_start:int, hour_end:int) -> pd.DataFrame: 
    sofa_input_df = block_df[['encounter_block','hospitalization_id','block_vent_start_dttm']].copy()
    sofa_input_df['start_dttm'] = sofa_input_df['block_vent_start_dttm'] + pd.Timedelta(hours=hour_start)
    sofa_input_df['end_dttm'] = sofa_input_df['block_vent_start_dttm'] + pd.Timedelta(hours=hour_end)
    sofa_input_df.drop(columns=['block_vent_start_dttm'], inplace=True)
    sofa_input_polars_df = pl.from_pandas(sofa_input_df) 
    sofa_df = compute_sofa_polars(
        data_directory = helper.config['clif'],
        cohort_df = sofa_input_polars_df, # id, start_time, end_time  (local time)
        filetype = "parquet",
        id_name="encounter_block",
        timezone="America/New_York",
        extremal_type = 'worst',
        fill_na_scores_with_zero=False,
        remove_outliers=True
    )
    sofa_df = sofa_df.to_pandas()#Convert form polars
    
    #Re-assign missingness (the function assigns 0 to missing values but we want N/A).
    sofa_missing_mask = (sofa_df['sofa_cv_97'].isna() |\
                         sofa_df['sofa_coag'].isna() |\
                         sofa_df['sofa_liver'].isna() |\
                         sofa_df['sofa_resp'].isna() |\
                         sofa_df['sofa_cns'].isna() |\
                         sofa_df['sofa_renal'].isna() )
    sofa_df['sofa_total'] = np.where(sofa_missing_mask,
                                     np.nan,
                                     sofa_df['sofa_total'])
    print(f"Empty SOFA scores ({hour_start}-{hour_end}): {sum(sofa_missing_mask)}")
    return sofa_df[['encounter_block','sofa_total']]

#SOFA score for 0 to 4 hours
temp_sofa_df = my_sofa_calculator(0,4)
temp_sofa_df.rename(columns={'sofa_total':'sofa_0_4h'},inplace=True)
block_df = pd.merge(
    block_df,
    temp_sofa_df,
    on='encounter_block',
    how='left'
)
del temp_sofa_df

#SOFA score for 0 to 24 hours
temp_sofa_df = my_sofa_calculator(0,24)
temp_sofa_df.rename(columns={'sofa_total':'sofa_0_24h'},inplace=True)
block_df = pd.merge(
    block_df,
    temp_sofa_df,
    on='encounter_block',
    how='left'
)
del temp_sofa_df

#SOFA score for -20 to 4 hours
temp_sofa_df = my_sofa_calculator(-20,4)
temp_sofa_df.rename(columns={'sofa_total':'sofa_-20_4h'},inplace=True)
block_df = pd.merge(
    block_df,
    temp_sofa_df,
    on='encounter_block',
    how='left'
)
del temp_sofa_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

📢 Starting SOFA score computation with Polars
📢 Data directory: /nfs/roberts/project/pi_sj692/shared/mimicclif
📢 Cohort size: 32147 rows
📢 Grouping by: encounter_block
📢 Standardizing cohort datetime columns to America/New_York with nanosecond precision
📢 Standardizing 2 datetime column(s) to America/New_York with time unit ns
📢 Successfully standardized 2 datetime column(s)
📢 Loading data for 32127 unique hospitalization(s)
📢 Loading labs data...
📢 Standardizing 1 datetime column(s) to America/New_York with time unit ns
📢 Successfully standardized 1 datetime column(s)
📢 Loading vitals data...
📢 Standardizing 1 datetime column(s) to America/New_York with time unit ns
📢 Successfully standardized 1 datetime column(s)
📢 Loading patient assessments data...
📢 Standardizing 1 datetime column(s) to America/New_York with time unit ns
📢 Successfully standardized 1 datetime column(s)
📢 Loading respiratory support data...
📢 Standardizing 1 datetime column(s) to America/New_York with time unit ns


/nfs/roberts/project/pi_sj692/gcr22/clifenv/lib/python3.12/site-packages/clifpy/utils/sofa_polars.py:685: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  meds = meds.join_asof(


📢 Collecting vitals...
📢 Collecting assessments...
📢 Collecting respiratory...
📢 Collecting medications...
📢 Preparing data for combination...
📢 Combining collected data sources...
📢 Applying outlier removal...
📢 Aggregating extremal values in long format...
📢 Pivoting aggregated data to wide format...
📢 Imputing PaO2 from SpO2...
📢 Calculating concurrent P/F ratios...
📢 Standardizing 1 datetime column(s) to America/New_York with time unit ns
📢 Successfully standardized 1 datetime column(s)
📢 Standardizing 1 datetime column(s) to America/New_York with time unit ns
📢 Successfully standardized 1 datetime column(s)
📢 Aggregating concurrent P/F ratios by encounter_block...
📢 Freeing memory from intermediate DataFrames...
📢 Computing SOFA scores...
📢 SOFA computation complete. Result shape: 32046 rows x 27 columns
Empty SOFA scores (0-4): 26685
📢 Starting SOFA score computation with Polars
📢 Data directory: /nfs/roberts/project/pi_sj692/shared/mimicclif
📢 Cohort size: 32147 rows
📢 Grouping 

/nfs/roberts/project/pi_sj692/gcr22/clifenv/lib/python3.12/site-packages/clifpy/utils/sofa_polars.py:796: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  po2_with_fio2 = po2_df.join_asof(


📢 Standardizing 1 datetime column(s) to America/New_York with time unit ns
📢 Successfully standardized 1 datetime column(s)
📢 Loading respiratory support data...
📢 Standardizing 1 datetime column(s) to America/New_York with time unit ns
📢 Successfully standardized 1 datetime column(s)
📢 Loading and converting medication data...
📢 Standardizing 1 datetime column(s) to America/New_York with time unit ns
📢 Successfully standardized 1 datetime column(s)
📢 Standardizing 1 datetime column(s) to America/New_York with time unit ns
📢 Successfully standardized 1 datetime column(s)
📢 Collecting individual data sources before combining...
📢 Collecting labs...


/nfs/roberts/project/pi_sj692/gcr22/clifenv/lib/python3.12/site-packages/clifpy/utils/sofa_polars.py:685: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  meds = meds.join_asof(


📢 Collecting vitals...
📢 Collecting assessments...
📢 Collecting respiratory...
📢 Collecting medications...
📢 Preparing data for combination...
📢 Combining collected data sources...
📢 Applying outlier removal...
📢 Aggregating extremal values in long format...
📢 Pivoting aggregated data to wide format...
📢 Imputing PaO2 from SpO2...
📢 Calculating concurrent P/F ratios...
📢 Standardizing 1 datetime column(s) to America/New_York with time unit ns
📢 Successfully standardized 1 datetime column(s)
📢 Standardizing 1 datetime column(s) to America/New_York with time unit ns
📢 Successfully standardized 1 datetime column(s)
📢 Aggregating concurrent P/F ratios by encounter_block...
📢 Freeing memory from intermediate DataFrames...
📢 Computing SOFA scores...
📢 SOFA computation complete. Result shape: 32115 rows x 27 columns
Empty SOFA scores (0-24): 19354
📢 Starting SOFA score computation with Polars
📢 Data directory: /nfs/roberts/project/pi_sj692/shared/mimicclif
📢 Cohort size: 32147 rows
📢 Grouping

/nfs/roberts/project/pi_sj692/gcr22/clifenv/lib/python3.12/site-packages/clifpy/utils/sofa_polars.py:796: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  po2_with_fio2 = po2_df.join_asof(


📢 Standardizing 1 datetime column(s) to America/New_York with time unit ns
📢 Successfully standardized 1 datetime column(s)
📢 Loading respiratory support data...
📢 Standardizing 1 datetime column(s) to America/New_York with time unit ns
📢 Successfully standardized 1 datetime column(s)
📢 Loading and converting medication data...
📢 Standardizing 1 datetime column(s) to America/New_York with time unit ns
📢 Successfully standardized 1 datetime column(s)


In [ ]:
#FIRST (sort-of) SAVING POINT
path = os.path.join(helper.path_out, "block_compiled_1b.parquet")
block_df.to_parquet(path)
del path

## Elixhauser
Using package comorbidipy with Quan et al mappings and Van Walraven weights

Outputs both unadjusted and age adjusted.

Note that tis uses either ICD 9 or ICD 10 codes for any given encounter_block. There was a hard switch at some point so there are actually NO encounters with both ICD codes mixed.

**This needs to be switched over to use CLIF table use the new version of CLIF-MIMIC generates them.**

In [ ]:
#Elixhauser
import pandas as pd
import os
import pthelperfunctions as helper
import comorbidipy

block_df = helper.load_data('output','block_compiled_1b')
diag_df = helper.load_clif_table('hospital_diagnosis',hosp_list=block_df['hospitalization_id'].tolist())

#filter out primary diagnosis
diag_df = diag_df[diag_df['diagnosis_primary'] != 1] #Remove primary diagnosis

#Merge with block data
diag_df = pd.merge(
        block_df[['encounter_block','hospitalization_id','admission_year','age']],
        diag_df[['hospitalization_id','diagnosis_code','diagnosis_code_format']],
        on='hospitalization_id',
        how='left'
    )

#Create separate data frames for each ICD version
diag_9_df = diag_df[diag_df['diagnosis_code_format'] == 'ICD9CM'].drop_duplicates().reset_index() 
diag_10_df = diag_df[diag_df['diagnosis_code_format'] == 'ICD10CM'].drop_duplicates().reset_index()
diag_9_df = diag_9_df[['encounter_block','diagnosis_code','age']]
diag_10_df = diag_10_df[['encounter_block','diagnosis_code','age']]

#Check how many encnounterblocks are in both datasets and report it as an issue.
duplicate_icds_n = sum(diag_9_df['encounter_block'].isin(diag_10_df['encounter_block']))
print(f"Number of encnounters with comorbidities in both ICD codes: {duplicate_icds_n}")

#Run it for ICD 9
elix_9_df = comorbidipy.comorbidity(
    diag_9_df,
    id='encounter_block',
    code='diagnosis_code',
    age='age',
    score = 'elixhauser',
    icd = 'icd9',
    variant = 'quan',
    weighting = 'vw',#Van Walraven weights
)

#Run it for ICD 10
elix_10_df = comorbidipy.comorbidity(
    diag_10_df,
    id='encounter_block',
    code='diagnosis_code',
    age='age',
    score = 'elixhauser',
    icd = 'icd10',
    variant = 'quan',
    weighting = 'vw',#Van Walraven weights
)

elix_9_df.head()

In [ ]:
#Concatenate both types
elix_df = pd.concat(
    [elix_10_df[['encounter_block','comorbidity_score','age_adj_comorbidity_score']],elix_9_df[['encounter_block','comorbidity_score','age_adj_comorbidity_score']]],
    ignore_index=True
)
#If any encounters have both types of diagnostic codes it will keep the ICD 10 code only. (first)
elix_df.drop_duplicates(subset='encounter_block',keep='first',inplace=True)

#Rename columns for our convention
elix_df.rename(columns={'comorbidity_score':'elixhauser','age_adj_comorbidity_score':'elixhauser_age_adj'},inplace=True)

block_df = pd.merge(
    block_df,
    elix_df,
    on='encounter_block',
    how='left'
)

#Fill values for empty encnounters.
#Justified because ALL encounters have ICD codes, missing implies no comorbidities because we removed the primary admisison diagnosis.
block_df = block_df.fillna(value={'elixhauser':0,'elixhauser_age_adj':0})

In [ ]:
#SECOND SAVING POINT
path = os.path.join(helper.path_out, "block_compiled_2.parquet")
block_df.to_parquet(path)
del path

## Hourly data analysis

Uses hourly data frame built by the CLIF Mobilization code

In [ ]:
#Add Hourly Block by Block Data
import pandas as pd
import os
import pytz
import statistics
from datetime import datetime
from datetime import timedelta
import numpy as np
import pthelperfunctions as helper

#Load up the hourly block final data
hourly_df = helper.load_data('mob','final_df_w_criteria',folder='intermediate')
hourly_df['encounter_block'] = hourly_df['encounter_block'].astype(str)
hourly_df = hourly_df.sort_values(['encounter_block', 'time_from_vent'])#Ensure proper chronological order

block_final = helper.load_data('output','block_compiled_2')
block_final['encounter_block'] = block_final['encounter_block'].astype(str)

#Load and filter Hospitalizations data from CLIF
hosp_df = helper.load_clif_table('hospitalization')
hosp_df = hosp_df[hosp_df['patient_id'].isin(block_final['patient_id'])].reset_index()

#Aadd encounter_block to the hospitalization data frame.
hosp_df = hosp_df.sort_values('admission_dttm')
hosp_df = hosp_df.merge(
    block_final[['patient_id','encounter_block','block_vent_start_dttm']].drop_duplicates(),
    on='patient_id',
    how='left').reset_index()

#Load assessments
rass_df = helper.load_clif_table('patient_assessments',
                                 hosp_list=block_final['hospitalization_id'].tolist())
rass_df = rass_df[rass_df['assessment_category'] == 'RASS']
rass_df.rename(columns={'numerical_value': 'RASS'}, inplace=True)

#Remove the ones not in cohort, add encounter_block and vent start time as reference
rass_df = rass_df.merge(hosp_df[['hospitalization_id','encounter_block','block_vent_start_dttm']],on='hospitalization_id', how='right').reset_index()

#Add date and time column
rass_df['recorded_date'] = rass_df['recorded_dttm'].dt.date
rass_df['recorded_hour'] = rass_df['recorded_dttm'].dt.hour
rass_df = rass_df[['encounter_block', 'recorded_date', 'recorded_hour', 'RASS']].copy().drop_duplicates()
hourly_df['recorded_date'] = pd.to_datetime(hourly_df['recorded_date'])
rass_df['recorded_date'] = pd.to_datetime(rass_df['recorded_date'])
hourly_df['recorded_hour'] = hourly_df['recorded_hour'].astype(int)

### RASS Data
From CLIF Patient assessment table and filled into the hourly blocks.

In [ ]:
#Add RASS data to hourly blocks
hourly_df = pd.merge_ordered(
    hourly_df,
    rass_df,
    on=['encounter_block', 'recorded_date', 'recorded_hour'],
    how='left')
del rass_df

#Forward fill RASS
hourly_df['RASS'] = (
    hourly_df
    .groupby(['encounter_block'])['RASS']
    .transform(lambda x: x.ffill(limit=12)) #Note this is because it was already sorted above when the hourly DF was called.
)

#Add labels to divide into the chunks we want
hourly_df['time_from_vent'] = hourly_df['time_from_vent'].astype(int)
hourly_df['time_window'] = hourly_df['time_from_vent'].apply(helper.classify_time_window)

#Define coma
hourly_df['coma'] = hourly_df['RASS'] < -2
hourly_df['coma'] = hourly_df['coma'].astype(int)

#Group by encounter block and by time window, calculate total time in coma
grouped_coma_df = (
    hourly_df.groupby(['encounter_block', 'time_window'])['coma']
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)

#Rename columns for clarity
grouped_coma_df.rename(columns={
    '0-24h': 'coma_0_24h',
    '24-48h': 'coma_24_48h',
    '48-72h': 'coma_48_72h'
}, inplace=True)
grouped_coma_df.drop(columns=['prior24h'], inplace=True, errors='ignore')

block_final = pd.merge(
    block_final,
    grouped_coma_df,
    on='encounter_block',
    how='left'
)

del grouped_coma_df

### Pressor Data
From the hourly blocks data frame already built.

In [ ]:
#Filter by time than Group by encounter block ONLY, calculate max pressor dose
pressor_df = hourly_df[hourly_df['time_from_vent'] <= 4]
grouped_pressor_df = (
    pressor_df.groupby('encounter_block')['ne_calc_max']
    .max()
    .reset_index()
)

#Rename columns for clarity
grouped_pressor_df.rename(columns={'ne_calc_max': 'pressor_0_4h',}, inplace=True)

#Convert to boolean
#Note that the hourly block data does not extend to prior to IMV initiation. For that, we use the sofa_score.py output.
grouped_pressor_df['pressor_0_4h'] = grouped_pressor_df['pressor_0_4h'] > 0


block_final = pd.merge(
    block_final,
    grouped_pressor_df,
    on='encounter_block',
    how='left'
)

del pressor_df, grouped_pressor_df

### Paralytics Data
From the hourly blocks data frame already built.

In [ ]:
#Group by encounter block and by time window, calculate sum of paralytics flags
grouped_para_df = (
    hourly_df.groupby(['encounter_block', 'time_window'])['paralytics_flag']
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)

#Rename columns for clarity
grouped_para_df.rename(columns={
    '0-24h': 'paralytics_0_24h',
    '24-48h': 'paralytics_24_48h',
    '48-72h': 'paralytics_48_72h'
}, inplace=True)
grouped_para_df.drop(columns=['prior24h'], inplace=True, errors='ignore')

#Convert to boolean
grouped_para_df['paralytics_0_24h'] = grouped_para_df['paralytics_0_24h'] >= 4
grouped_para_df['paralytics_24_48h'] = grouped_para_df['paralytics_24_48h'] >= 4
grouped_para_df['paralytics_48_72h'] = grouped_para_df['paralytics_48_72h'] >= 4

block_final = pd.merge(
    block_final,
    grouped_para_df,
    on='encounter_block',
    how='left'
)

del grouped_para_df

### Yellow Eligibility Data
From the hourly blocks data frame already built.

In [ ]:
#ELIGIBILITY TOTAL HOURS
## Group by encounter block and by time window, calculate total time in yellow eligibility all hours
grouped_yellow_df = (
    hourly_df.groupby(['encounter_block', 'time_window'])['any_yellow_or_green_no_red_all_hours']
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)

#Rename columns for clarity
grouped_yellow_df.rename(columns={
    '0-24h': 'yellow_0_24h',
    '24-48h': 'yellow_24_48h',
    '48-72h': 'yellow_48_72h'
}, inplace=True)

block_final = pd.merge(
    block_final,
    grouped_yellow_df,
    on='encounter_block',
    how='left'
)

del grouped_yellow_df

In [ ]:
#ELIGIBILITY FIRST 2-HOURS CONSECUTIVE
# Create new data frame
yellow_df = hourly_df[['encounter_block','time_from_vent','any_yellow_or_green_no_red_all_hours']].copy()
yellow_df.rename(columns={'any_yellow_or_green_no_red_all_hours':'yellow'}, inplace=True)
yellow_df['time_from_vent'] = yellow_df['time_from_vent'].astype(int)

#Two consecutive hours (note this relies on the hourly_df being sorted which we do above.
yellow_df['yellow_2'] = (yellow_df['yellow'].shift(periods=1, fill_value=0) == 1) & yellow_df['yellow']
yellow_df = yellow_df[yellow_df['yellow_2'] & (yellow_df['time_from_vent'] > 1)] #The >1 is to prevent accidental counting of the previous encounter block.

#Group and get the first hour
grouped_yellow_df = (
    yellow_df.groupby('encounter_block')['time_from_vent']
    .min()
    .reset_index()
)
grouped_yellow_df.rename(columns={'time_from_vent': 'yellow_time_eligibility_2h'}, inplace=True)

block_final = pd.merge(
    block_final,
    grouped_yellow_df[['encounter_block','yellow_time_eligibility_2h']],
    on='encounter_block',
    how='left'
)
block_final['yellow_2h_0_72h'] = (block_final['yellow_time_eligibility_2h'] <= 72).astype(bool)

del grouped_yellow_df, yellow_df

### Ventilator Data
From the hourly blocks data frame already built.

In [ ]:
###VENT FREE DAYS
#If dead, 0
#Otherwise uses the last hour of IMV on the hourly data_frame
vent_df = hourly_df[['encounter_block','time_from_vent','hourly_on_vent']]
vent_df['time_from_vent'] = vent_df['time_from_vent'].astype(int)
vent_df['hourly_on_vent'] = vent_df['hourly_on_vent'].astype(bool)
#Keep only values within 28-days and on-vent
vent_df = vent_df[(vent_df['time_from_vent'] <= 28*24) & vent_df['hourly_on_vent'] ]
#Get the MAX hour and merge it into DF
last_vent_df = (
    vent_df.groupby('encounter_block')['time_from_vent']
    .max()
    .reset_index()
)
last_vent_df['time_from_vent'] = last_vent_df['time_from_vent'].astype(int)
last_vent_df.rename(columns={'time_from_vent':'last_hour_on_vent'}, inplace=True)
block_final = pd.merge(
    block_final,
    last_vent_df,
    on='encounter_block',
    how='left'
)
del last_vent_df
del vent_df
#Get an 1 for patients alive at 28-days.
block_final['alive28'] = block_final['date_of_death'].isna() | ((block_final['date_of_death'] - block_final['block_vent_start_dttm']).dt.total_seconds() >= (28*24*60*60))
block_final['alive28'] = block_final['alive28'].astype(int)
#Calcute VFD
block_final['vent_free_days'] = block_final['alive28'] * (28 - block_final['last_hour_on_vent']/24)
block_final = block_final.drop(columns=['alive28','last_hour_on_vent'])

#REINTUBATIONS
vent_df = hourly_df[['encounter_block','time_from_vent','hourly_on_vent']] #Note it is already sorted in the proper order

#Intubation count for hospitalization only, not including other hospitalizations.
def count_intubations(series):
    """Counts the number of re-intubations as 0->1 transitions."""
    return ((series.shift(periods=1, fill_value=0) == 1) & (series == 0)).sum()

intubation_count_df = (
    vent_df
    .groupby('encounter_block')['hourly_on_vent']
    .apply(count_intubations)
    .reset_index()
    .rename(columns={'hourly_on_vent': 'intubation_count'})
)
block_final = pd.merge(
    block_final,
    intubation_count_df,
    on='encounter_block',
    how='left'
)
block_final['reintubation'] = (block_final['intubation_count'] > 1)

del hourly_df, intubation_count_df


print(f"Block Length: {len(block_final)}")
print(f"Unique Encounter Block: {block_final['encounter_block'].nunique()}")

#THIRD SAVING POINT
path = os.path.join(helper.path_out, "block_compiled_3.parquet")
block_final.to_parquet(path)
del path

## Basic Math

Calculations for easily calculated/derived variables from our compiled variables.

In [ ]:
#FINALIZE DATA FRAME FOR STATISTICAL ANALYSIS
import pandas as pd
import os
import pytz
import statistics
from datetime import datetime
from datetime import timedelta
import numpy as np
import pthelperfunctions as helper

#Load file
path = os.path.join(helper.path_out, "block_compiled_3.parquet")
block_final = pd.read_parquet(path)
del path

#Change relevant DTTM values to hours/days
block_final['imv_to_discharge_days'] = (block_final['discharge_dttm'] - block_final['block_vent_start_dttm']).dt.total_seconds()/(24*3600)
block_final['imv_to_end_hours'] = (block_final['block_vent_end_dttm'] - block_final['block_vent_start_dttm']).dt.total_seconds()/(3600)
block_final['adm_to_imv_hours'] = (block_final['block_vent_start_dttm'] - block_final['admission_dttm']).dt.total_seconds()/3600
block_final['imv_to_death_days'] = (block_final['death_dttm'] - block_final['block_vent_start_dttm']).dt.total_seconds()/(24*3600)
block_final['adm_to_discharge_days'] = (block_final['discharge_dttm'] - block_final['block_first_vital_dttm']).dt.total_seconds()/(24*3600)
block_final['icu_to_imv_hours'] = (block_final['block_vent_start_dttm'] - block_final['icu_in_dttm']).dt.total_seconds()/(3600) #Positive if in ICU first before IMV.
block_final['Time_first_PT'] = (block_final['pt_post_imv_dttm'] - block_final['block_vent_start_dttm']).dt.total_seconds()/3600
block_final['Time_last_PT'] = (block_final['pt_pre_imv_dttm'] - block_final['block_vent_start_dttm']).dt.total_seconds()/3600

#Add in a dichotomized outcomes variables
block_final['pt_ever'] = block_final['pt_post_imv_dttm'].notna()
block_final['pt_post48_IMV'] = block_final['Time_first_PT'].notna() & (block_final['Time_first_PT'] <= 48)
block_final['pt_pre24_IMV'] = block_final['Time_last_PT'].notna() & (block_final['Time_last_PT'] >= -24)
block_final['yellow_ever'] = block_final['yellow_time_eligibility'].notna()
block_final['yellow_post48_IMV'] = block_final['yellow_ever'] & (block_final['yellow_time_eligibility'] <= 48)
block_final['extubated_at_pt'] = block_final['imv_to_end_hours'] <= block_final['Time_first_PT']
block_final['is_dead'] = block_final['is_dead'].astype(bool)
block_final['pt_between_ICU_IMV'] = block_final['Time_first_PT'] < (-1*block_final['icu_to_imv_hours'])

# Add Hospital mortality: TRUE if Death_dttm < discharge_dttm or (discharge category is hospice or dead) 
block_final["is_dead_hosp"] = (
    (block_final["death_dttm"] <= block_final["discharge_dttm"]) |
    (block_final["discharge_category"].str.lower().isin(["Hospice", "Expired"]))
)
#30-day mortality
block_final['is_dead_30'] = (block_final['date_of_death'] - block_final['block_vent_start_dttm']).dt.total_seconds() <= (30*24*60*60)
#365-day mortality
block_final['is_dead_365'] = (block_final['date_of_death'] - block_final['block_vent_start_dttm']).dt.total_seconds() <= (365*24*60*60)

print(f"Block Length: {len(block_final)}")
print(f"Unique Encounter Block: {block_final['encounter_block'].nunique()}")

#FOURTH SAVING POINT
path = os.path.join(helper.path_out, "block_compiled_4.parquet")
block_final.to_parquet(path)
del path

#Column check point
path = os.path.join(helper.path_out, "column_checkpoint.txt")
with open(path, "w") as text_file:
    for col_name in block_final.sort_index(axis=1).columns.tolist():
        text_file.write(col_name + '\n')
del path

## Clustering of Categorical Data

In [ ]:
##CLUSTERING OF CATEGORICAL DATA
import pthelperfunctions as helper
import pandas as pd
import numpy as np

path = os.path.join(helper.path_out, "block_compiled_4.parquet")
df = pd.read_parquet(path)
del path

#Individualy and manually cluster all the categorical variables we want to cluster
print("LANGUAGE PRE:")
print(df['language'].value_counts(dropna=False))
keep = {"English","Spanish"}
set_mask = df['language'].notna() & (~df['language'].eq("None"))
df["language"] = np.where(df["language"].isin(keep), df["language"], "Other")
df["language"] = np.where(set_mask, df['language'], None)
print("LANGUAGE POST:")
print(df['language'].value_counts(dropna=False)) #Print results

print("RACE PRE:")
print(df['race_category'].value_counts(dropna=False))
keep = {"White", "Black or African American"}
set_mask = df['race_category'].notna() & (~df['race_category'].eq("Unknown"))
df["race_category"] = np.where(df["race_category"].isin(keep), df["race_category"], "Other")
df["race_category"] = np.where(set_mask, df['race_category'], None)
print("RACE POST:")
print(df['race_category'].value_counts(dropna=False)) #Print results

print("INSURANCE PRE:")
print(df['insurance'].value_counts(dropna=False))
keep = {"Medicare", "Medicaid","Private"}
set_mask = df['insurance'].notna() & (~df['insurance'].eq("Unknown"))
df["insurance"] = np.where(df["insurance"].isin(keep), df["insurance"], "None/Other")
df["insurance"] = np.where(set_mask, df['insurance'], None)
print("INSURANCE POST:")
print(df['insurance'].value_counts(dropna=False)) #Print results

#This just converts "Unknown" to None for better missingness tracking.
set_mask = df['ethnicity_category'].notna() & (~df['ethnicity_category'].eq("Unknown"))
df["ethnicity_category"] = np.where(set_mask, df['ethnicity_category'], None)

print("ICU TYPE PRE:")
print(df['ICU_type'].value_counts(dropna=False))
mapping = {
    "Medical Intensive Care Unit (MICU)": "Medical ICU",
    "Medical/Surgical Intensive Care Unit (MICU/SICU)": "Medical ICU",
    "Intensive Care Unit (ICU)": "Medical ICU", 
    "Surgical Intensive Care Unit (SICU)": "Surgical ICU",
    "Coronary Care Unit (CCU)": "Cardiac ICU",
    "Cardiac Vascular Intensive Care Unit (CVICU)": "Cardiac ICU",
    "Trauma SICU (TSICU)": "Other",
    "Neuro Surgical Intensive Care Unit (Neuro SICU)": "Other"
}
df['ICU_type'] = df['ICU_type'].map(mapping)
df['ICU_type'] = np.where(df['ICU_type'].notna(), df['ICU_type'], None)
print("ICU TYPE POST:")
print(df['ICU_type'].value_counts(dropna=False))

print("DISCHARGE PRE:")
print(df['discharge_category'].value_counts(dropna=False))
mapping = {
    "Home": "Home",
    "Against Medical Advice (AMA)": "Home",
    "Assisted Living": "Home",
    "Hospice": "Hospice", "Expired": "Expired",
    "Skilled Nursing Facility (SNF)": "Rehabilitation",
    "Acute Inpatient Rehab Facility": "Rehabilitation",
    "Psychiatric Hospital": "Other",
    "Acute Care Hospital": "Other",
    "Long Term Care Hospital (LTACH)": "Other",
    "Other": "Other", "Missing": "Missing"
}
df['discharge_category'] = df['discharge_category'].map(mapping)
print("DISCHARGE POST:")
print(df['discharge_category'].value_counts(dropna=False))

'''
#THIS WAS REMOVED AND REPLACED WITH THE CLIF CATEGORIZED VARIABLE
print("ADMISSION SOURCE PRE:")
print(df['admission_source'].value_counts(dropna=False))
##THIS IS A PLACEHOLDER, WE NEED TO UPDATE MIMIC to CLIF MAPPINGS TO GET ADMISSION_TYPE_CATEGORY instead.
mapping = {
    "EW EMER.": "Emergency/Urgent",
    "URGENT": "Emergency/Urgent",
    "EU OBSERVATION": "Observation",
    "OBSERVATION ADMIT": "Observation",
    "DIRECT OBSERVATION": "Observation",
    "SURGICAL SAME DAY ADMISSION": "Surgery",
    "DIRECT EMER.": "Direct",
    "ELECTIVE": "Direct",
    "AMBULATORY OBSERVATION": "Observation",
}
df['admission_source'] = df['admission_source'].map(mapping)
print("ADMISSION SOURCE POST:")
print(df['admission_source'].value_counts(dropna=False))
'''
print("MINIC Race PRE:")
print(df['mimic_race'].value_counts(dropna=False))
mapping = {
    "WHITE":"White",
    "UNKNOWN":"Missing",
    "BLACK/AFRICAN AMERICAN":"Black/African American",
    "OTHER":"Other",
    "UNABLE TO OBTAIN":"Missing",
    "WHITE - OTHER EUROPEAN":"White",
    "HISPANIC/LATINO - PUERTO RICAN":"Hispanic/Latino",
    "ASIAN":"Other",
    "ASIAN - CHINESE":"Other",
    "HISPANIC/LATINO - DOMINICAN":"Hispanic/Latino",
    "WHITE - RUSSIAN":"White",
    "PATIENT DECLINED TO ANSWER":"Missing",
    "HISPANIC OR LATINO":"Hispanic/Latino",
    "BLACK/CAPE VERDEAN":"Black/African American",
    "BLACK/CARIBBEAN ISLAND":"Black/African American",
    "PORTUGUESE":"White",
    "BLACK/AFRICAN":"Black/African American",
    "ASIAN - SOUTH EAST ASIAN":"Other",
    "WHITE - EASTERN EUROPEAN":"White",
    "ASIAN - ASIAN INDIAN":"Other",
    "AMERICAN INDIAN/ALASKA NATIVE":"Other",
    "WHITE - BRAZILIAN":"White",
    "HISPANIC/LATINO - GUATEMALAN":"Hispanic/Latino",
    "HISPANIC/LATINO - SALVADORAN":"Hispanic/Latino",
    "NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER":"Other",
    "HISPANIC/LATINO - CUBAN":"Hispanic/Latino",
    "ASIAN - KOREAN":"Other",
    "HISPANIC/LATINO - HONDURAN":"Hispanic/Latino",
    "HISPANIC/LATINO - MEXICAN":"Hispanic/Latino",
    "MULTIPLE RACE/ETHNICITY":"Other",
    "SOUTH AMERICAN":"Hispanic/Latino",
    "HISPANIC/LATINO - CENTRAL AMERICAN":"Hispanic/Latino",
    "HISPANIC/LATINO - COLUMBIAN":"Hispanic/Latino"
}
df['mimic_race'] = df['mimic_race'].map(mapping)
print("MIMIC Race POST:")
print(df['mimic_race'].value_counts(dropna=False))

#FIFTH SAVING POINT
path = os.path.join(helper.path_out, "block_compiled_5.parquet")
df.to_parquet(path)
del path


## Organize Columns and Summarize

In [ ]:
import pandas as pd
import scipy.stats as stats
import numpy as np
import os
import pthelperfunctions as helper

path = os.path.join(helper.path_out, "block_compiled_5.parquet")
df = pd.read_parquet(path)
column_order = pd.read_csv(os.path.join("..","config","column_def.csv"))
my_cols = column_order['name'].tolist()
column_order = column_order.set_index('name')

df = df.drop_duplicates(subset=['encounter_block'], keep='first')
df = df[my_cols]
path = os.path.join(helper.path_out, "block_for_stats.parquet")
df.to_parquet(path)
path = os.path.join(helper.path_out, "block_for_stats.csv")
df.to_csv(path)
del path

#Exclusion criteria
print(f"To be excluded based on PT Prior 24 hours to IMV: {sum(df['pt_pre24_IMV'])}")
df = df[~df['pt_pre24_IMV']]

#Convert outcome to a categorical column
early_col = "pt_post48_IMV"
df["early_PT"] = np.where(df[early_col], "early_PT", "no_early_PT")
n_total = df["encounter_block"].count()
n_early = df[early_col].sum()
n_not = n_total - n_early

file_path = os.path.join(helper.path_out,"table1.csv")
if os.path.exists(file_path):
    os.remove(file_path)

with open(file_path, mode="w") as file:
    file.write(f",,Overall,Early PT, No Early PT, P-value, Missing")
    for col in my_cols:
        lab = column_order.loc[col,"description"]
        if col == 'encounter_block':
            file.write(f"\nN,,{n_total},{n_early},{n_not},")
        elif df[col].dtype == 'object' or pd.api.types.is_string_dtype(df[col]):
            file.write(f"\n{lab}")
            cats = df[col].unique()
            p_df = pd.pivot_table(df[["encounter_block",col,"early_PT"]], index=col, columns="early_PT", values="encounter_block", aggfunc='count')
            chi2_stat, p_value, dof, expected = stats.chi2_contingency(p_df.to_numpy())
            for cc in cats:
                if cc: #Because there is a NAN category
                    cc_all = round(100*p_df.loc[cc,].sum()/df[col].count(),1)
                    cc_early = round(100*p_df.loc[cc,'early_PT'].sum()/p_df['early_PT'].sum(),1)
                    cc_not = round(100*p_df.loc[cc,'no_early_PT'].sum()/p_df['no_early_PT'].sum(),1)
                    file.write(f"\n,{cc},{cc_all}%,{cc_early}%,{cc_not}%")
            file.write(f", {round(p_value,5)}")
        elif df[col].dtype == "bool" or df[col].dtype == "boolean":
            if df[col].sum(skipna=True) > 0:#Pivot table does not work if there are no true values
                sub_df = df[df[col].notna()]
                sub_df['flag'] = np.where(sub_df[col],"TRUE", "FALSE")
                p_df = pd.pivot_table(sub_df[["encounter_block","flag","early_PT"]], index="flag", columns="early_PT", values="encounter_block", aggfunc='count')
                chi2_stat, p_value, dof, expected = stats.chi2_contingency(p_df.to_numpy())
                cc_all = round(100*p_df.loc["TRUE",].sum()/sub_df[col].count(),1)
                cc_early = round(100*p_df.loc["TRUE",'early_PT'].sum()/p_df['early_PT'].sum(),1)
                cc_not = round(100*p_df.loc["TRUE",'no_early_PT'].sum()/p_df['no_early_PT'].sum(),1)
                file.write(f"\n{lab},,{cc_all}%,{cc_early}%,{cc_not}%,{round(p_value,5)}")
            else:
                file.write(f"\n{lab},,0.00%,0.00%,0.00%,N/A")
        elif pd.api.types.is_numeric_dtype(df[col]):
            cc_all = df[col].dropna()
            cc_early = df.loc[df[early_col], col]
            cc_early = cc_early.dropna()
            cc_not = df.loc[~df[early_col], col]
            cc_not = cc_not.dropna()
            rank_stat, p_value = stats.ranksums(cc_early.tolist(), cc_not.tolist())
            file.write(f"\n{lab} (Med & IQR),,{round(cc_all.median(),2)}  ({round(cc_all.quantile(0.25),2)} - {round(cc_all.quantile(0.75),2)})")
            file.write(f",{round(cc_early.median(),2)}  ({round(cc_early.quantile(0.25),2)} - {round(cc_early.quantile(0.75),2)})")
            file.write(f",{round(cc_not.median(),2)}  ({round(cc_not.quantile(0.25),2)} - {round(cc_not.quantile(0.75),2)})")
            file.write(f",{round(p_value,5)}")
        else:
            file.write(f"\n{lab},ERROR,,,,,")
        #Missing data column
        mis_pct = round(100*sum(df[col].isna())/n_total,2)
        file.write(f",{mis_pct}%")
            

## CIF Graph

In [ ]:
import matplotlib.pyplot as plt

#Create a new lists
yellow_list = df['yellow_time_eligibility_2h'].sort_values().dropna()
pt_list = df['Time_first_PT'].sort_values().dropna()

# Plot CIF
plt.figure(figsize=(8, 6))
plt.step(yellow_list, np.arange(yellow_list.size), color='orange')
plt.step(pt_list, np.arange(pt_list.size), color='blue')
plt.xlabel("Hours from IMV Initiation")
plt.xlim(0, 72)
plt.ylabel("Encounter Blocks")
plt.title("Physioliogic Readiness versus PT Consult Order CIF")
plt.legend()

#Save and show
path = os.path.join(helper.path_out, "CIF_Yellow_v_PT.png")
plt.savefig(path)
plt.show()